# 1. 불러오기와 기본 점검

In [1]:
# 도구 불러오기, 그래프 한글 폰트 설정
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"   # 그래프 한글 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False      # 마이너스 기호 깨짐 방지

In [2]:
# 최종 표 불러오기 / 분기 코드를 연도와 분기로 나누기 (20214 다음이 20221이라 숫자 계산용으로 분리)
df = pd.read_csv("../data/processed/cafe_full_2021_2025.csv")
print(df.shape)
df["연도"] = df["기준_년분기_코드"] // 10
df["분기"] = df["기준_년분기_코드"] % 10
print(df[["기준_년분기_코드", "연도", "분기"]].drop_duplicates().head(6))

(21572, 154)
      기준_년분기_코드    연도  분기
0         20241  2024   1
1087      20242  2024   2
2171      20243  2024   3
3259      20244  2024   4
4348      20251  2025   1
5425      20252  2025   2


In [3]:
# 숫자 칸만 골라서 0값·음수 점검 / 개업률·폐업률 100% 초과 행 확인
num = df.select_dtypes("number")
zero = (num == 0).sum()
print("0이 하나라도 있는 칸:", (zero > 0).sum(), "개")
print(zero.sort_values(ascending=False).head(15))
print("음수 자리 수:", (num < 0).sum().sum())
print("폐업률 100 초과 행:", (df["폐업_률"] > 100).sum())
print("개업률 100 초과 행:", (df["개업_율"] > 100).sum())

0이 하나라도 있는 칸: 127 개
아파트_가구_수            21469
공항_수                21241
버스_터미널_수            21201
중학교_수               21161
고등학교_수              21044
백화점_수               21021
초등학교_수              21006
종합병원_수              20822
슈퍼마켓_수              20062
대학교_수               19770
극장_수                19664
숙박_시설_수             19065
일반_병원_수             18822
유치원_수               18751
남성연령대_10_직장_인구_수    18202
dtype: int64
음수 자리 수: 0
폐업률 100 초과 행: 1
개업률 100 초과 행: 0


# 2. 점포당 매출 만들기

In [4]:
# 점포당 매출 = 분기 매출 ÷ 카페 수 (카페 수는 프랜차이즈 포함인 유사_업종_점포_수)
df["점포당_매출"] = df["당월_매출_금액"] / df["유사_업종_점포_수"]
print("분모가 0인 행:", (df["유사_업종_점포_수"] == 0).sum())
print("빈칸:", df["점포당_매출"].isna().sum())
print(df["점포당_매출"].describe().round(0))

분모가 0인 행: 0
빈칸: 0
count        21572.0
mean      21293221.0
std       26034661.0
min            472.0
25%        5487453.0
50%       13202538.0
75%       27781846.0
max      359955677.0
Name: 점포당_매출, dtype: float64


- 점포당 분기 매출 중앙값 1,320만 원 (카페 한 곳 월 약 440만 원), 평균 2,129만 원 → 큰 상권이 평균을 끌어올림
- 당월_매출_금액은 분기 합계로 판단: 시청역 8번(카페 40곳, 133억)이 월 기준이면 카페당 월 3.3억이라 비현실적, 1건당 매출 중앙값 약 7,400원
- 최솟값 472원 = 면목역 2번 2024년 1분기(카페 14곳, 분기 매출 6,604원·13건) → 추정 실패 행
- 점포당 분기 매출 100만 원 미만 1,059행(4.9%) → 변화율 계산 때 제외 기준 검토